1. Importar librerías

In [25]:
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_20newsgroups

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.decomposition import NMF

from sklearn.metrics.pairwise import cosine_similarity

2. Cargar el dataset

In [26]:
dataset = fetch_20newsgroups(
    subset='all',
    remove=('headers','footers','quotes')
)

documents = dataset.data

3. Convertir los artículos en una matriz TF-IDF

In [27]:
vectorizer = TfidfVectorizer(

    stop_words='english',

    max_features=5000

)

tfidf = vectorizer.fit_transform(documents)

In [28]:
print('Dimensiones originales (Artículos x Palabras):', tfidf.shape)

Dimensiones originales (Artículos x Palabras): (18846, 5000)


La matriz `tfidf` tiene dimensiones `(18846, 5000)`. Esto significa que el dataset contiene **18,846 artículos** en total y cada uno está representado por **5,000 palabras clave** (las palabras más importantes seleccionadas por el vectorizador).

4. Aplicar NMF

In [29]:
nmf = NMF(max_iter=500, 

    n_components=20,

    random_state=42

)

W = nmf.fit_transform(tfidf)

H = nmf.components_

In [30]:
print('Dimensiones de W (Artículos x Temas):', W.shape)
print('Dimensiones de H (Temas x Palabras):', H.shape)

Dimensiones de W (Artículos x Temas): (18846, 20)
Dimensiones de H (Temas x Palabras): (20, 5000)


Al aplicar **NMF con 20 componentes (`n_components=20`)**, la matriz original gigante se comprimió y dividió en dos más pequeñas:
- **Matriz W:** De tamaño `(18846, 20)`. Ahora cada uno de los 18,846 artículos está resumido en solo **20 temas** en lugar de 5,000 palabras.
- **Matriz H:** De tamaño `(20, 5000)`. Nos indica qué peso tiene cada una de las 5,000 palabras dentro de los 20 temas descubiertos.

5. Calcular la similitud entre artículos

In [31]:
similarity = cosine_similarity(W)

6. Función de recomendación

In [32]:
def recomendar(indice, n=5):

    similitudes = similarity[indice]

    indices = np.argsort(similitudes)[::-1]

    indices = indices[1:n+1]

    return indices

7. Probar el sistema

In [35]:
articulo = 120

# 1. Identificar el tema principal del artículo leído usando la matriz W
tema_principal = W[articulo].argmax()
peso_tema = W[articulo][tema_principal]

# 2. Extraer las palabras clave de ese tema usando la matriz H y el vectorizador
palabras = vectorizer.get_feature_names_out()
top_palabras_idx = H[tema_principal].argsort()[::-1][:10]
palabras_clave = [palabras[i] for i in top_palabras_idx]

print("================ ARTÍCULO LEÍDO ================\n")
print(documents[articulo], "\n")
print("*" * 70)
print(f"--> El NMF detectó que el TEMA DOMINANTE es el #{tema_principal}")
print(f"--> PALABRAS CLAVE que hicieron el Match: {', '.join(palabras_clave)}")
print("*" * 70, "\n")

print("\n================ ARTÍCULOS RECOMENDADOS ================\n")

# Para cada artículo recomendado, mostrar también su porcentaje de similitud
for i in recomendar(articulo):
    similitud = similarity[articulo][i] * 100
    tema_rec = W[i].argmax() # Ver si el tema principal es el mismo
    
    print("-" * 70)
    print(f"Recomendación con {similitud:.2f}% de similitud (Tema principal detectado: #{tema_rec})")
    print("-" * 70)
    print(documents[i])


================ ARTÍCULO LEÍDO ================

This might be a silly question but I have to ask it anyway. I am in
the process of purchasing an EISA/VL Bus 486 DX2-66 computer and I
found two places that sell machines that have what I want and have the
same price. The first is Ares and they use a Cache motherboard (that's
the brand of the motherboard) with OPTI chip set, the other is Micron
(formerly Edge Technology) and they use the Micronics EISA/VLB
motherboard.

I said that this might be a silly question since I believe that
Micronics is a very well known motherboard manufacturer while I never
heard of Cache! I am however leaning towards the Ares machine because
my impression is that they are known for building good, solid machines
and they have good tech support (24 hr, 7 days/wk), and a better
warrantee (2 years).  Micron, on the other hand, seems to have
recently aquired Edge Technologies and I'm not sure how much I should
trust the company.

I would REALLY appreciate any inp

### 8. Conclusiones y Resultados
El sistema de recomendación basado en NMF ha demostrado ser muy efectivo. Como se observa en la prueba del paso anterior:
- El artículo leído trata sobre **hardware de computadoras**, específicamente sobre la compra de un PC con placa base EISA/VLB, monitores y chips de video.
- Los artículos recomendados mantienen una gran coherencia temática, sugiriendo discusiones sobre monitores monocromáticos, tarjetas de video VRAM, resoluciones de pantalla y problemas de visualización en monitores VGA.
- El uso de la factorización **NMF** junto con la **similitud del coseno** ha permitido capturar exitosamente los "temas latentes" de los artículos (en este caso, tecnología y hardware) y recuperar los artículos más afines al interés actual del usuario de forma puramente no supervisada.
